In [1]:
import sys
from numbers import Real
from pathlib import Path
from html import escape

import pandas as pd
from IPython.display import HTML, display

sys.path.append("../")

from evaluation.classes import ResponseGroundednessWrapper
from evaluation.eval import groundedness, load_tests
from evaluation.reporting import export_groundedness_report_to_markdown
from pipelines.faq_pipeline import FaqPipeline


/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Users/ayeustsihneyeu/PythonProjects/hybrid-rag-system/.hybrag/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def _node_metadata(item):
    if hasattr(item, "node"):
        return getattr(item.node, "metadata", {}) or {}
    return getattr(item, "metadata", {}) or {}


def _node_text(item):
    node = getattr(item, "node", item)
    if hasattr(node, "get_content"):
        return node.get_content()
    return getattr(node, "text", "") or ""


def _node_score(item):
    score = getattr(item, "score", None)
    return round(float(score), 4) if isinstance(score, Real) else None


def _preview(text, max_len=220):
    cleaned = " ".join(str(text).split())
    if len(cleaned) <= max_len:
        return cleaned
    return cleaned[: max_len - 3] + "..."


def _render_metrics(metrics):
    metrics_html = "".join(
        f"""
        <div style='padding:12px 14px;border:1px solid #d0d7de;border-radius:10px;background:#f6f8fa;'>
            <div style='font-size:12px;color:#57606a;text-transform:uppercase;letter-spacing:.04em;'>{escape(name)}</div>
            <div style='font-size:24px;font-weight:700;color:#24292f;'>{value:.3f}</div>
        </div>
        """
        for name, value in metrics.items()
    )
    display(HTML(f"<div style='display:grid;grid-template-columns:repeat(2, minmax(0, 1fr));gap:12px;margin:12px 0 18px;'>{metrics_html}</div>"))


def _render_question_report(test, row, doc_rows, index, total):
    header_html = f"""
    <div style='margin:28px 0 10px;'>
        <div style='font-size:12px;color:#57606a;text-transform:uppercase;letter-spacing:.08em;'>Question {index}/{total}</div>
        <div style='font-size:22px;font-weight:700;margin:6px 0 10px;color:#24292f;'>{escape(test.question)}</div>
        <div style='padding:12px 14px;background:#fff8c5;border:1px solid #e3d279;border-radius:10px;'>
            <div style='font-size:12px;color:#57606a;text-transform:uppercase;letter-spacing:.04em;margin-bottom:6px;'>Relevant docs</div>
            <div style='font-size:16px;font-weight:600;color:#24292f;'>{escape(str(test.relevant_docs))}</div>
        </div>
    </div>
    """
    display(HTML(header_html))
    _render_metrics({"groundedness": row["groundedness"]})
    display(HTML(f"<p style='margin:8px 0 4px;'><strong>Reference answer:</strong> {escape(str(row['reference_answer']))}</p>"))
    display(HTML(f"<p style='margin:8px 0 12px;'><strong>Generated answer:</strong> {escape(str(row['generated_answer']))}</p>"))
    display(pd.DataFrame(doc_rows))
    return {
        "question": test.question,
        "relevant_docs": list(test.relevant_docs),
        "retrieved_docs": row["retrieved_docs"],
        "reference_answer": row["reference_answer"],
        "generated_answer": row["generated_answer"],
        "groundedness": row["groundedness"],
        "doc_rows": doc_rows,
    }


async def evaluate_all_groundedness(limit=None, markdown_path=None):
    tests = load_tests()
    selected_tests = tests[:limit] if limit is not None else tests
    pipeline = FaqPipeline()
    rows = []
    question_reports = []

    for index, test in enumerate(selected_tests, start=1):
        pipeline_result = pipeline.run(test.question)
        nodes = pipeline_result["nodes"]
        contexts = [_node_text(item) for item in nodes]
        answer = pipeline_result["answer"]

        score = await groundedness(
            ResponseGroundednessWrapper(
                contexts=contexts,
                answer=answer,
            )
        )

        retrieved_docs = []
        doc_rows = []
        for rank, item in enumerate(nodes, start=1):
            metadata = _node_metadata(item)
            section_id = metadata.get("section_id")
            retrieved_docs.append(section_id)
            doc_rows.append(
                {
                    "rank": rank,
                    "section_id": section_id,
                    "title": metadata.get("section_title", "-"),
                    "rerank_score": _node_score(item),
                    "preview": _preview(_node_text(item)),
                }
            )

        row = {
            "question": test.question,
            "reference_answer": test.reference_answer,
            "generated_answer": answer,
            "relevant_docs": test.relevant_docs,
            "retrieved_docs": retrieved_docs,
            "groundedness": score,
        }
        rows.append(row)

        question_report = _render_question_report(
            test,
            row,
            doc_rows,
            index=index,
            total=len(selected_tests),
        )
        question_report.update({"index": index, "total": len(selected_tests)})
        question_reports.append(question_report)

    report_df = pd.DataFrame(rows)
    summary_df = report_df[["groundedness"]].mean().to_frame(name="mean").T.round(3)

    display(HTML("<h2 style='margin:32px 0 12px;color:#24292f;'>Final metrics</h2>"))
    display(summary_df)

    if markdown_path is not None:
        saved_path = export_groundedness_report_to_markdown(
            report_df,
            question_reports,
            Path(markdown_path),
            notebook_path="notebooks/12_faq_groundedness_evaluating.ipynb",
        )
        display(HTML(f"<p style='margin:12px 0 0;color:#1a7f37;'>Markdown report saved to: <code>{escape(str(saved_path))}</code></p>"))

    return report_df


In [3]:
await evaluate_all_groundedness(markdown_path="../reports/faq_groundedness_evaluation.md")


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 19564.07it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,rank,section_id,title,rerank_score,preview
0,1,1,How to register with an email,8.5447,How to register with an email Open the registr...
1,2,2,How to register with Google or Facebook account,5.6311,How to register with Google or Facebook accoun...
2,3,49,How to provide shipping details?,5.2245,How to provide shipping details? If you have a...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 11725.71it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,rank,section_id,title,rerank_score,preview
0,1,2,How to register with Google or Facebook account,7.5312,How to register with Google or Facebook accoun...
1,2,3,Signing in with a password and email address,-6.3560,Signing in with a password and email address I...
2,3,1,How to register with an email,-8.5365,How to register with an email Open the registr...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9517.01it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

,rank,section_id,title,rerank_score,preview
0,1,49,How to provide shipping details?,4.6625,How to provide shipping details? If you have a...
1,2,90,I have been waiting a long time for my order d...,-0.0519,I have been waiting a long time for my order d...
2,3,61,How to order a parcel with HopShop Delivery,-0.2747,How to order a parcel with HopShop Delivery Wh...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 16793.59it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,rank,section_id,title,rerank_score,preview
0,1,50,I am buying many products — which delivery met...,5.7683,I am buying many products — which delivery met...
1,2,34,How to choose a delivery option,5.5872,How to choose a delivery option If you buy at ...
2,3,32,How to pay for orders made from multiple sellers,2.5484,How to pay for orders made from multiple selle...


,groundedness
mean,0.995


,question,reference_answer,generated_answer,relevant_docs,retrieved_docs,groundedness
0,How do I register for a HopShop account using ...,Open the registration page. Choose whether you...,"To register with your email address, open the ...",[1],"[1, 2, 49]",1.00
1,How can I sign up using my Google or Facebook ...,Open the sign in page.\nSelect signing in thro...,To sign up using your Google or Facebook accou...,[2],"[2, 3, 1]",1.00
2,How can I enable signing in with my email addr...,If you want to be able to sign in to your acco...,To enable signing in with your email address a...,[3],"[3, 1, 2]",1.00
3,How do I register for a HopShop account using ...,Open the registration page. Select whether you...,"To register with your phone number, open the r...",[4],"[4, 4, 1]",1.00
4,How can I download my personal data from my Ho...,"On the Copies of your data page, you can downl...",To download your personal data from your HopSh...,[5],"[5, 11, 6]",1.00
5,Am I allowed to have multiple HopShop accounts...,"Yes, you can have a few accounts, as long as y...","Yes, you can have multiple HopShop accounts as...",[6],"[6, 40, 46]",1.00
6,How do I close my HopShop account?,You can end your activity on HopShop by:\nwith...,You can close your HopShop account by withdraw...,[7],"[7, 16, 14]",1.00
7,How can I withdraw from my agreement with HopS...,\n Section_Id: 8,You can withdraw from your agreement with HopS...,[8],"[9, 7, 8]",1.00
8,Under what conditions can I withdraw from my a...,You can easily withdraw from the agreement wit...,You can withdraw from your agreement with HopS...,[9],"[9, 8, 12]",1.00
9,How can I withdraw from my agreement and what ...,"To withdraw from the agreement, submit this ap...","To withdraw from the agreement, submit the app...",[10],"[10, 9, 11]",1.00
